# Step 1 — Extract & Transform: Price Data and Technical Indicators (X_t)

This is the first working piece of the ETL pipeline from the architecture diagram. It covers the **left-most box** (External Data Sources → Yahoo Finance) and the first half of the **Transform** stage (Apache Spark, eventually — for now, plain pandas) that produces **X_t**, the technical/price feature vector that later gets fused with sentiment into F_t.

**Why start here?** Of the three external sources (Yahoo Finance, NewsAPI, Reddit), price data is the only one that needs no API key or account sign-up — `yfinance` talks to Yahoo Finance for free. That means this step has the fewest things that can go wrong, so it's the right place to build confidence before Step 2 introduces API keys.

By the end of this notebook you'll have a table with one row per trading day and columns `SMA_20`, `RSI_14`, `OBV`, `ADX_14` — that table, before any join with sentiment, **is** X_t.

## 1. Install the libraries we need

- **yfinance** — the Python wrapper around Yahoo Finance's price data. No account or API key required.
- **pandas / numpy** — already in Colab by default.
- **matplotlib** — to plot the result and visually sanity-check it. Always look at a feature before you trust it.

Run the cell below (`-q` just keeps the install log quiet).

In [ ]:
!pip install -q yfinance

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

pd.set_option("display.max_columns", None)
print("yfinance version:", yf.__version__)

## 2. Extract: pull raw OHLCV price data

"OHLCV" = **O**pen, **H**igh, **L**ow, **C**lose, **V**olume — the daily prices and trade volume. This is the rawest form of price data; every indicator below is *derived* from these five numbers.

Two design decisions that matter later, not just now:

- **Pull a long enough date range.** Indicators like a 20-day moving average or 14-day RSI need a "warm-up" period of history before their first valid value. Too short a range and your indicators are `NaN` the whole way through. We'll pull several years.
- **No look-ahead.** For any prediction date, you may only use data up to and including that date — never data from the future. Obvious here, but it's the most common bug in financial ML projects ("look-ahead bias"), and it's why every feature we eventually store gets tagged with the exact date it was computed as-of.

Pick any ticker you like — `AAPL` is the running example below.

In [ ]:
TICKER = "AAPL"
START = "2018-01-01"
END = "2024-12-31"

raw = yf.download(TICKER, start=START, end=END, auto_adjust=True, progress=False)

print(raw.shape)
raw.head()

## 3. Look before you trust it: basic data-quality checks

This is a small, by-hand preview of what the "Great Expectations" box in the architecture diagram does at scale: before raw data is allowed downstream, it gets checked. Three quick checks:

1. Any missing values?
2. Any duplicate dates?
3. Does the date index look like real trading days (no unexplained multi-week gaps)?

Make this a habit on every dataset before building features on top of it — one bad row silently poisons every indicator computed from it.

In [ ]:
print("Missing values per column:")
print(raw.isna().sum())

print("\nDuplicate dates:", raw.index.duplicated().sum())

gaps = raw.index.to_series().diff().dt.days
print("\nLargest gaps between consecutive trading days:")
print(gaps.sort_values(ascending=False).head())

## 4. Transform: Simple Moving Average (SMA) — trend

**What it is:** the average closing price over the last N days, recalculated every day.

**Why it's in X_t:** a single day's close is noisy. The SMA smooths that noise and tells the model the *direction* prices have been trending — something the raw price alone doesn't communicate well to a model that sees data one row at a time. 20 days (≈ one trading month) is a common short/medium window.

**Formula**, for day *t*:

`SMA_20(t) = mean(Close(t), Close(t-1), ..., Close(t-19))`

In pandas, "the last N rows, recomputed at every row" is exactly `.rolling(window=N).mean()`.

In [ ]:
df = raw.copy()
df["SMA_20"] = df["Close"].rolling(window=20).mean()

df[["Close", "SMA_20"]].tail(10)

## 5. Transform: Relative Strength Index (RSI) — momentum

**What it is:** a 0–100 score comparing the size of recent up-days to recent down-days. Above 70 is conventionally "overbought," below 30 "oversold."

**Why it's in X_t:** SMA tells you *direction*; RSI tells you *speed and exhaustion* — momentum a moving average can't capture. The actual design goal when choosing a feature set is to minimize redundancy between features, not just add more of them — SMA and RSI give the model two genuinely different views of the same price series.

**Formula** (14-day, Wilder's smoothing — the standard version):

1. day-over-day price change → split into gains and losses
2. exponentially-smoothed average gain and average loss over the window (Wilder's method, not a flat average)
3. RS = average gain / average loss
4. RSI = 100 − 100 / (1 + RS)

We compute it by hand below instead of importing a ready-made indicator library, specifically so every line maps back to one of the four steps above.

In [ ]:
def compute_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, min_periods=period, adjust=False).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

df["RSI_14"] = compute_rsi(df["Close"], period=14)

df[["Close", "RSI_14"]].tail(10)

## 6. Transform: On-Balance Volume (OBV) — volume confirmation

**What it is:** a running total that adds the day's volume when price closed up, and subtracts it when price closed down.

**Why it's in X_t:** a price move on heavy trading volume is a stronger signal than the same move on thin volume. OBV is how the model gets access to *conviction* behind a move. It's also the only one of the four indicators that uses the `Volume` column — without it, volume never reaches the model in any form.

**Formula:**

```
OBV(t) = OBV(t-1) + Volume(t)   if Close(t) > Close(t-1)
OBV(t) = OBV(t-1) - Volume(t)   if Close(t) < Close(t-1)
OBV(t) = OBV(t-1)               if Close(t) == Close(t-1)
```

In [ ]:
def compute_obv(close: pd.Series, volume: pd.Series) -> pd.Series:
    direction = np.sign(close.diff()).fillna(0)
    return (direction * volume).cumsum()

df["OBV"] = compute_obv(df["Close"], df["Volume"])

df[["Close", "Volume", "OBV"]].tail(10)

## 7. Transform: Average Directional Index (ADX) — trend strength

**What it is:** a 0–100 score that, unlike RSI, doesn't say *up* or *down* — it says *how strongly trending* the market currently is, in either direction. Below ~20 usually means choppy/sideways; above ~25–30 means a real trend is in place.

**Why it's in X_t:** this is the piece that tells the model how much to *trust* the trend and momentum signals above. A strong RSI reading during a strong trend (high ADX) means something different from the same RSI reading in a choppy market (low ADX). It's the most involved formula here because it's built in layers, each reusing the one before it:

1. directional movement (+DM, −DM): how much of today's high/low range pushed in each direction
2. True Range (TR): the real size of today's range, accounting for gaps from yesterday's close
3. smooth +DM, −DM, and TR over the window → +DI and −DI
4. DX = how far apart +DI and −DI are, as a percentage of their sum
5. ADX = a smoothed average of DX — the final score

Looks like a lot, but each line below maps directly onto one of those five steps.

In [ ]:
def compute_adx(high: pd.Series, low: pd.Series, close: pd.Series, period: int = 14) -> pd.Series:
    up_move = high.diff()
    down_move = -low.diff()

    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)
    plus_dm = pd.Series(plus_dm, index=high.index)
    minus_dm = pd.Series(minus_dm, index=high.index)

    prev_close = close.shift(1)
    true_range = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low - prev_close).abs(),
    ], axis=1).max(axis=1)

    atr = true_range.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    plus_di = 100 * (plus_dm.ewm(alpha=1/period, min_periods=period, adjust=False).mean() / atr)
    minus_di = 100 * (minus_dm.ewm(alpha=1/period, min_periods=period, adjust=False).mean() / atr)

    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di)
    adx = dx.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    return adx

df["ADX_14"] = compute_adx(df["High"], df["Low"], df["Close"], period=14)

df[["Close", "ADX_14"]].tail(10)

## 8. Deal with the warm-up period

Every indicator above needs some history before its first value means anything — the first 19 rows have no `SMA_20`, the first 13 have no `RSI_14` / `ADX_14`. Right now those show up as `NaN`.

Two honest options, and the choice matters for how the model eventually gets trained:

- **Drop the warm-up rows.** Simple and safe; you lose the first few weeks of your pulled range.
- **Pull extra lookback.** Fetch e.g. 30 extra days *before* the date range you actually care about purely so the indicators are already warmed up, then trim back to the intended range.

We pulled several years of data, so dropping the first ~20 rows costs us almost nothing — that's what we'll do here.

In [ ]:
print("Rows before dropping warm-up NaNs:", len(df))
df_clean = df.dropna().copy()
print("Rows after:", len(df_clean))

df_clean[["SMA_20", "RSI_14", "OBV", "ADX_14"]].isna().sum()

## 9. This table is X_t

Keep only the engineered indicator columns — drop the raw OHLCV, since the model consumes *engineered* features, not raw prices directly. One row per trading day, in exactly the shape X_t needs to be in to later get fused with S_t and H_t into F_t.

In [ ]:
X_t = df_clean[["SMA_20", "RSI_14", "OBV", "ADX_14"]].copy()
X_t.index.name = "date"
X_t["ticker"] = TICKER

X_t.tail(10)

## 10. Sanity-check it visually

Numbers in a table can hide bugs that are obvious the moment you look at a chart — a flat-lined indicator, or one that's offset by a day because of a `shift()` mistake. Always plot a new feature before trusting it.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)

axes[0].plot(df_clean.index, df_clean["Close"], label="Close")
axes[0].plot(df_clean.index, df_clean["SMA_20"], label="SMA_20")
axes[0].set_title(f"{TICKER} — Close vs SMA_20")
axes[0].legend()

axes[1].plot(df_clean.index, df_clean["RSI_14"], color="darkorange")
axes[1].axhline(70, color="red", linestyle="--", linewidth=0.8)
axes[1].axhline(30, color="green", linestyle="--", linewidth=0.8)
axes[1].set_title("RSI_14 (70 = overbought, 30 = oversold)")

axes[2].plot(df_clean.index, df_clean["ADX_14"], color="purple")
axes[2].axhline(25, color="gray", linestyle="--", linewidth=0.8)
axes[2].set_title("ADX_14 (above ~25 = trending market)")

plt.tight_layout()
plt.show()

## 11. Load (for now): save the result

In the full architecture, this table eventually lands in PostgreSQL, written by an Airflow-scheduled Spark job. We don't have that infrastructure yet — that's a deliberate later step, not an oversight. For now "Load" just means saving the feature table to Google Drive so it can be reused without recomputing it every time.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = "/content/drive/MyDrive/StockPricePrediction/features"
os.makedirs(SAVE_DIR, exist_ok=True)

out_path = f"{SAVE_DIR}/X_t_{TICKER}.csv"
X_t.to_csv(out_path)
print("Saved:", out_path)

## What you just built, and what's next

A working Extract → Transform → Load path for the price side of the pipeline. The engineering decisions you actually made, not just the code you ran:

- started with the source that has no auth friction (Yahoo Finance), so the *logic* could be validated before infrastructure friction got added on top
- validated raw data before building anything on it
- chose four indicators deliberately — trend, momentum, volume, trend-strength — instead of four variations on the same idea
- handled the warm-up NaN problem explicitly instead of letting it silently propagate downstream
- visually sanity-checked the result before trusting it
- saved the output as a clearly-labeled, reusable artifact

**Step 2 (next):** the text side — pulling headlines from NewsAPI and posts from Reddit (first time we'll deal with API keys), then running FinBERT over the cleaned text to get **S_t** (sentiment score) and **H_t** (entropy of FinBERT's probability distribution) — the two numbers that feed the entropy gate. Same approach: one cell, one concept, tested as we go.